In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import os, getpass, gc, copy, torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from torchmetrics.classification import MulticlassAccuracy, MulticlassRecall, MulticlassPrecision

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device)}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
df_path = os.path.join(script_dir, "Datasets", "Building Energy Use.csv")
df = pd.read_csv(df_path)
df = df.dropna()
df["target_class"] = df["target_class"].map({"High Use":1,"Medium Use":2,"Low Use":3})

x = df.drop(columns="target_class").values
x_scaler = StandardScaler()
y = df["target_class"]
x = x_scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.786, random_state=22)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

user_account = getpass()

num_features = x_train.shape[1]
num_classes = len(y)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim=num_features, out_dim=num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=16),
            nn.LeakyReLU(),
            nn.BatchNorm1d(16),
            
            nn.Linear(in_features=16, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=28, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),   
            
            nn.Linear(in_features=47, out_features=52),
            nn.ELU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.8),   

            nn.Linear(in_features=52, out_features=68),
            nn.ELU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.3),

            nn.Linear(in_features=68, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),   

            nn.Linear(in_features=42, out_features=21),
            nn.LeakyReLU(),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=out_dim)
        )     

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

losses = []
accuracies = []
def grid_search(rates_list, epochs_list, seed_val):
    best_state = None
    loss_fn = nn.CrossEntropyLoss()

    print(f"\n== Leaderboard ==")    
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15} | {'Val Accuracy':<15}")
    print("="*80)

    for lr in rates_list:
        for count in epochs_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = MultiClassClassifier().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds,y_train.squeeze().long())

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_logits = model(x_test)
                test_loss = loss_fn(test_logits, y_test.squeeze().long())  
                accuracy = MulticlassAccuracy(num_classes=num_classes).to(device)
                precision = MulticlassPrecision(num_classes=num_classes).to(device)
                recall = MulticlassRecall(num_classes=num_classes).to(device)
                test_acc = accuracy(torch.sigmoid(test_logits),y_test.squeeze().long()).item()
                losses.append(test_loss)
                accuracies.append(test_acc)

            print(f"{lr:<15} | {count:<15} | {test_loss:<15} | {test_acc:<15}")
            best_state = copy.deepcopy(model.state_dict())
            del model, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies):.2f}")
    print(f"● Worst Loss = {max(losses):.2f} | Worst Accuracy = {min(accuracies):.2f}")
    print(f"● Precision Score = {precision(torch.sigmoid(test_logits), y_test.squeeze().long()).item():.2f}")
    print(f"● Recall Score = {recall(torch.sigmoid(test_logits), y_test.squeeze().long()).item():.2f}")
    return best_state

epoch_count_list = [980,1230,780,810]
learning_rates = [0.1, 0.001, 0.01, 0.02]
seed_val = float(input("-- Enter a Value for the Seed : ").strip())
best_state = grid_search(epochs_list=epoch_count_list, rates_list=learning_rates, seed_val=seed_val)
torch.save(best_state, f"Building Energy Use Predictor Parameters")

fresh_model = MultiClassClassifier().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

conversion = {
    1:"High Use",
    2:"Medium Use",
    3:"Low Use"
}

while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip()
    if user_name.lower() == "exit" or not user_name:
        print(f"Exiting Interactive Loop ...")
        break

    floor_area_m2 = float(input("-- Enter the Floor Area in Meters Squared : ").strip())
    occupants = float(input("-- Enter the Number of Occupants : ").strip())
    outside_temp_c = float(input("-- Enter the Outside Tempeature in Celsius : ").strip())
    insulation_score = float(input("-- Enter the Insulation Score : ").strip())
    hvac_hours_daily = float(input("-- Enter the Daily HVAC Hours : ").strip())

    raw_array = np.array([floor_area_m2, occupants, outside_temp_c, insulation_score, hvac_hours_daily])
    scaled_tensor = torch.tensor(x_scaler.transform(raw_array), dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = fresh_model(scaled_tensor)
        probibilty = torch.softmax(logits, dim=1)

    print(f"Predicted Building Energy Use Status for you, {user_name.title()} = {conversion.get(probibilty, 1)}")
    session_inputs.append(f"{user_name.title()} ( {outside_temp_c} ° C )")
    session_predictions.append(probibilty)

try:
    onedrive = input("-- Do you use One Drive or Not ( Yes / No ) :").strip().title()
    if onedrive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Building Energy Use Predictor Details.txt"
    elif onedrive == "No":
        desktop_path = f"C:/Users/{user_account}/Desktop/Building Energy Use Predictor Details.txt"
except BaseException as e:
    print(f"Invalid Input | Error has Occured = {e}")
try:
    with open(desktop_path, "w", encoding="utf-8") as file:
            file.write(f"== Building Energy Use Predictor Details ==\n")   
            file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
            file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
            file.write(f"● Best Loss = {min(losses):.3f}\n")
            file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"\nSuccessfully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")    